# 05 - End-to-End Orchestration

This notebook demonstrates the complete workflow: loading invoices, extracting data, validating, and submitting to Fakturoid.

In [4]:
from src.document_processor import DocumentProcessor
from src.ai_extractor import AIExtractor, InvoiceData
from src.fakturoid_client import FakturoidClient
from src.config import config
from src.agent import InvoiceProcessingAgent
import json
from pathlib import Path
from datetime import datetime

In [5]:
# Initialize agent
agent = InvoiceProcessingAgent(config)

print("Invoice Processing Agent initialized")
print(f"Mode: {config.processing.mode}")
print(f"Auto-submit: {config.processing.auto_submit}")
print(f"Invoices directory: {config.directories.invoices}")

Invoice Processing Agent initialized
Mode: manual
Auto-submit: False
Invoices directory: /Users/pavelzverina/AiProjects/fakturoid/data/invoices


In [6]:
# Test connections
print("Testing connections...")
if agent.test_connections():
    print("✓ All connections successful!")
else:
    print("✗ Connection test failed. Check your credentials.")

2025-10-08 20:49:08,036 - src.agent - INFO - Testing Fakturoid connection...
2025-10-08 20:49:08,226 - src.agent - INFO - All connections successful


Testing connections...
✓ All connections successful!


In [7]:
# Process all invoices with manual review
print("="*60)
print("PROCESSING ALL INVOICES")
print("="*60)

# Get list of files first
files = agent.doc_processor.list_invoice_files()
print(f"\nFound {len(files)} invoice files")

if len(files) == 0:
    print(f"\nNo invoice files found in: {agent.invoices_dir}")
    print("Please add PDF or image files to process.")
else:
    # Process batch with review enabled
    results = agent.process_batch(review=True, max_files=None)
    
    print(f"\n{'='*60}")
    print("PROCESSING SUMMARY")
    print(f"{'='*60}")
    submitted = sum(1 for r in results if r['status'] == 'submitted')
    extracted = sum(1 for r in results if r['status'] == 'extracted')
    errors = sum(1 for r in results if r['status'] == 'error')
    
    print(f"Total invoices: {len(results)}")
    print(f"Submitted: {submitted}")
    print(f"Extracted (pending review): {extracted}")
    print(f"Failed: {errors}")

2025-10-08 20:49:23,871 - src.agent - INFO - Processing 4 invoice files
2025-10-08 20:49:23,872 - src.agent - INFO - Processing file: Alien Isolation.pdf


PROCESSING ALL INVOICES

Found 4 invoice files


2025-10-08 20:49:28,598 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 20:49:28,609 - src.agent - INFO - Extracted data from Alien Isolation.pdf
2025-10-08 20:49:28,610 - src.agent - INFO - Processing file: FP20250158 - OrderSummary202509013059566415002039.png



EXTRACTED INVOICE DATA
Invoice Number: 786940972572357
Issue Date: 2025-10-02
Supplier: Sony Interactive Entertainment Network Europe Limited
Total Amount: 207.25 CZK

Line Items:
  1. Alien: Isolation (Game) - 1 x 0



2025-10-08 20:49:34,073 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 20:49:34,075 - src.agent - INFO - Extracted data from FP20250158 - OrderSummary202509013059566415002039.png
2025-10-08 20:49:34,075 - src.agent - INFO - Processing file: OpenAI-Invoice-3D6B9186-0031.pdf



EXTRACTED INVOICE DATA
Invoice Number: 3059566415002039
Issue Date: 2025-08-25
Supplier: Haiming One Store Store
Total Amount: 105.07 CZK

Line Items:
  1. Tuya WiFi Smart IR Remote Control - 1 x 105.07



2025-10-08 20:49:38,605 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 20:49:38,607 - src.agent - INFO - Extracted data from OpenAI-Invoice-3D6B9186-0031.pdf
2025-10-08 20:49:38,608 - src.agent - INFO - Processing file: google-workspace-5369924648.pdf



EXTRACTED INVOICE DATA
Invoice Number: 3D6B9186-0031
Issue Date: 2025-10-06
Supplier: OpenAI, LLC
Total Amount: 20.0 USD
Due Date: 2025-10-06

Line Items:
  1. ChatGPT Plus Subscription Oct 6 – Nov 6, 2025 - 1 x 20.0



2025-10-08 20:49:43,839 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 20:49:43,841 - src.agent - INFO - Extracted data from google-workspace-5369924648.pdf
2025-10-08 20:49:43,842 - src.agent - INFO - Batch processing complete: 0 submitted, 0 errors out of 4 total



EXTRACTED INVOICE DATA
Invoice Number: 5369924648
Issue Date: 2025-09-30
Supplier: Google Cloud EMEA Limited
Total Amount: 8.1 EUR

Line Items:
  1. Google Workspace Business Starter - 1 x 0


PROCESSING SUMMARY
Total invoices: 4
Submitted: 0
Extracted (pending review): 4
Failed: 0


In [8]:
# Show detailed results
if 'results' in locals() and results:
    print("\nDetailed Results:")
    for result in results:
        print(f"\n{'-'*60}")
        print(f"File: {result['file']}")
        print(f"Status: {result['status']}")
        
        if result['status'] == 'submitted':
            if result.get('extracted_data'):
                data = result['extracted_data']
                print(f"Invoice Number: {data.get('invoice_number', 'N/A')}")
                print(f"Supplier: {data.get('supplier_name', 'N/A')}")
                print(f"Amount: {data.get('total_amount', 'N/A')} {data.get('currency', 'CZK')}")
            if result.get('fakturoid_response'):
                print(f"Fakturoid ID: {result['fakturoid_response'].get('id')}")
                print(f"Fakturoid Number: {result['fakturoid_response'].get('number')}")
                print(f"URL: {result['fakturoid_response'].get('html_url')}")
        
        elif result['status'] == 'extracted':
            if result.get('extracted_data'):
                data = result['extracted_data']
                print(f"Invoice Number: {data.get('invoice_number', 'N/A')}")
                print(f"Supplier: {data.get('supplier_name', 'N/A')}")
                print(f"Amount: {data.get('total_amount', 'N/A')} {data.get('currency', 'CZK')}")
                print("⚠️  Awaiting manual review/approval")
        
        elif result['status'] == 'error':
            print(f"Error: {result.get('error', 'Unknown error')}")
else:
    print("No results to display. Run the previous cell first.")


Detailed Results:

------------------------------------------------------------
File: Alien Isolation.pdf
Status: extracted
Invoice Number: 786940972572357
Supplier: Sony Interactive Entertainment Network Europe Limited
Amount: 207.25 CZK
⚠️  Awaiting manual review/approval

------------------------------------------------------------
File: FP20250158 - OrderSummary202509013059566415002039.png
Status: extracted
Invoice Number: 3059566415002039
Supplier: Haiming One Store Store
Amount: 105.07 CZK
⚠️  Awaiting manual review/approval

------------------------------------------------------------
File: OpenAI-Invoice-3D6B9186-0031.pdf
Status: extracted
Invoice Number: 3D6B9186-0031
Supplier: OpenAI, LLC
Amount: 20.0 USD
⚠️  Awaiting manual review/approval

------------------------------------------------------------
File: google-workspace-5369924648.pdf
Status: extracted
Invoice Number: 5369924648
Supplier: Google Cloud EMEA Limited
Amount: 8.1 EUR
⚠️  Awaiting manual review/approval


In [9]:
# Process a single invoice with detailed steps
files = agent.doc_processor.list_invoice_files()

if files:
    test_file = files[0]
    print(f"Processing single invoice: {test_file.name}")
    print("="*60)
    
    # Use the agent's process_file method
    print("\n📄 Processing invoice...")
    result = agent.process_file(test_file, review=True)
    
    print(f"\n{'='*60}")
    print(f"Status: {result['status']}")
    print(f"{'='*60}")
    
    if result['status'] == 'error':
        print(f"\n✗ Error: {result['error']}")
    
    elif result['status'] == 'extracted':
        print("\n✓ Invoice data extracted successfully!")
        print("⚠️  Review the data above. To submit, set auto_submit=True")
    
    elif result['status'] == 'submitted':
        print("\n✓ Invoice submitted to Fakturoid!")
        if result.get('fakturoid_response'):
            resp = result['fakturoid_response']
            print(f"   ID: {resp.get('id')}")
            print(f"   Number: {resp.get('number')}")
            print(f"   URL: {resp.get('html_url')}")
else:
    print("No invoice files found.")
    print(f"Please add PDF or image files to: {agent.invoices_dir}")

2025-10-08 20:51:08,052 - src.agent - INFO - Processing file: Alien Isolation.pdf


Processing single invoice: Alien Isolation.pdf

📄 Processing invoice...


2025-10-08 20:51:13,046 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 20:51:13,048 - src.agent - INFO - Extracted data from Alien Isolation.pdf



EXTRACTED INVOICE DATA
Invoice Number: 786940972572357
Issue Date: 2025-10-02
Supplier: Sony Interactive Entertainment Network Europe Limited
Total Amount: 207.25 CZK

Line Items:
  1. Alien: Isolation (Game) - 1 x 0


Status: extracted

✓ Invoice data extracted successfully!
⚠️  Review the data above. To submit, set auto_submit=True


In [10]:
# Check processed invoices directory
processed_dir = config.directories.processed
if processed_dir.exists():
    processed_files = [f for f in processed_dir.glob("*") if f.is_file() and not f.name.startswith('.')]
    
    print(f"Processed files directory: {processed_dir}")
    print(f"Number of processed files: {len(processed_files)}")
    
    if processed_files:
        print("\nProcessed files (most recent first):")
        sorted_files = sorted(processed_files, key=lambda x: x.stat().st_mtime, reverse=True)
        for f in sorted_files[:10]:
            mtime = datetime.fromtimestamp(f.stat().st_mtime)
            print(f"  - {f.name} ({mtime.strftime('%Y-%m-%d %H:%M')})") 
        if len(processed_files) > 10:
            print(f"  ... and {len(processed_files) - 10} more")
    else:
        print("\n📁 No processed files yet")
else:
    print(f"Processed directory doesn't exist yet: {processed_dir}")
    print("It will be created when the first invoice is submitted.")

Processed files directory: /Users/pavelzverina/AiProjects/fakturoid/data/processed
Number of processed files: 0

📁 No processed files yet
